# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking**

My lane is content refresh prioritisation. The goal is to rank existing pages from highest to lowest refresh priority so an editor can decide which pages to review first.

The ranking can use signals such as content age, search performance, impressions, CTR, average position, content type, and recent performance trends.

Ranking fits this task because the editorial team has limited time. The goal is not simply to label every page as "refresh" or "do not refresh"; it is to decide which pages should be considered first.

In [1]:
print("Task type: Ranking")
print("Decision: which pages should an editor prioritise for refresh review?")

Task type: Ranking
Decision: which pages should an editor prioritise for refresh review?


## 2. Target or proxy

The target will initially be a **proxy for refresh priority**, rather than a claim that a page definitely needs refreshing.

I will use observed search-performance decline as the first proxy. A page whose recent performance is declining can be treated as a candidate for refresh review.

The proxy will be:

- `1` = declining page
- `0` = non-declining page

This is only a proxy because the available data does not directly tell us whether refreshing a page would cause its performance to improve. The eventual ranking should use multiple page-level signals rather than treating decline alone as the final decision.

In [2]:
from pathlib import Path
import pandas as pd

# Find the starter dataset whether the notebook is run from the repo root
# or from work/notebooks.
candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/content_refresh_anonymized.csv. "
        "Run this notebook from the repository."
    )

df = pd.read_csv(data_path)

# Sketch the proposed proxy target from the observed trend direction.
df["target_declining"] = (df["trend_direction"] == "down").astype(int)

print(f"Rows loaded: {len(df):,}")
print("Proxy target distribution:")
print(df["target_declining"].value_counts().sort_index())
print(f"\nDeclining rate: {df['target_declining'].mean():.1%}")

Rows loaded: 30,000
Proxy target distribution:
target_declining
0    13738
1    16262
Name: count, dtype: int64

Declining rate: 54.2%


## 3. Success metric

**Primary success metric: Precision@50**

Precision@50 measures the proportion of the 50 highest-ranked pages that are actually in the target/proxy group.

This matches the editorial workflow because the team has limited capacity. If the system recommends 50 pages for review, I care more about how useful those top 50 recommendations are than about correctly classifying every page in the dataset.

A successful ranking system should therefore put a high proportion of relevant refresh candidates near the top.

`Precision@50 = relevant pages in the top 50 / 50`

The supplied model results give a useful reference point: the fixed-rule baseline achieved **0.240 Precision@50**, while the random forest achieved **0.740 Precision@50**. This means the random forest put about 37 declining pages in its top 50, compared with about 12 for the baseline.

The goal is therefore to maximise Precision@50 while keeping the output useful as decision-support for editorial review.

In [3]:
# Precision@50 is the primary success metric for this ranking task.
K = 50
baseline_precision_at_50 = 0.240
random_forest_precision_at_50 = 0.740

print(f"Success metric: Precision@{K}")
print(f"Baseline Precision@{K}: {baseline_precision_at_50:.3f}")
print(f"Random forest Precision@{K}: {random_forest_precision_at_50:.3f}")
print(f"Baseline: about {round(baseline_precision_at_50 * K)} relevant pages in the top {K}")
print(f"Random forest: about {round(random_forest_precision_at_50 * K)} relevant pages in the top {K}")

Success metric: Precision@50
Baseline Precision@50: 0.240
Random forest Precision@50: 0.740
Baseline: about 12 relevant pages in the top 50
Random forest: about 37 relevant pages in the top 50


## 4. The unit of analysis, as a real dataframe

**One row = one content page's observed search-performance record.**

The unit of analysis is a single page. Each row contains page-level signals that could be used to decide whether that page should receive refresh attention.

I will show the actual dataframe below rather than describing the unit only conceptually. I will also show the proposed proxy target so that the ML framing is connected to the observations.

In [4]:
# One row represents one content page.
unit = df[
    [
        "content_type",
        "word_count",
        "content_age_days",
        "search_volume",
        "impressions_90d",
        "ctr",
        "avg_position",
        "position_tier",
        "trend_direction",
        "target_declining",
    ]
].copy()

print(f"Rows: {unit.shape[0]:,}")
print(f"Columns shown: {unit.shape[1]}")
print("\nOne row = one content page")
unit.head(10)

Rows: 30,000
Columns shown: 10

One row = one content page


,content_type,word_count,content_age_days,search_volume,impressions_90d,ctr,avg_position,position_tier,trend_direction,target_declining
0,keyword article,3221.0,187,10.0,3803,0.76,10.6,striking,down,1
1,keyword article,2481.0,445,90.0,15320,0.05,20.3,page_3_5,down,1
2,keyword article,3515.0,141,0.0,12581,0.09,36.5,page_3_5,down,1
3,keyword article,NaN,463,10.0,11751,0.49,6.2,page_1,stable,0
4,keyword article,2803.0,263,0.0,19140,0.13,44.0,page_3_5,down,1
5,keyword article,3080.0,147,720.0,3970,0.03,8.5,page_1,down,1
6,keyword article,3059.0,90,0.0,20,0.00,7.0,page_1,down,1
7,keyword article,NaN,445,590.0,1724,0.06,21.2,page_3_5,stable,0
8,keyword article,3807.0,90,0.0,32574,0.09,46.0,page_3_5,down,1
9,keyword article,NaN,257,0.0,1240,0.16,4.9,page_1,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could be:

`IF trend_direction == "down" THEN prioritise the page.`

The problem is that this treats every declining page as equally important. It also ignores other signals such as search demand, impressions, CTR, average position, content age, and content type.

ML can combine these signals to produce a ranking that is more useful for prioritisation than a single hand-written threshold.

This matters because the editorial team has limited time. The output supports a real content action: an editor can use the ranked list to decide which pages to review first.

The model is decision-support, not causal proof. A high-ranked page is a page worth investigating; it is not a guarantee that refreshing the page will improve performance.

In [5]:
print("Fixed-rule approach:")
print("IF trend_direction == 'down' THEN prioritise the page")

print("\nML approach:")
print("Combine multiple page-level signals to rank pages by refresh priority")

print("\nObserved model comparison:")
print(f"Baseline Precision@50: {baseline_precision_at_50:.3f}")
print(f"Random forest Precision@50: {random_forest_precision_at_50:.3f}")
print("The learned model improves the top-50 ranking over the fixed-rule baseline.")

Fixed-rule approach:
IF trend_direction == 'down' THEN prioritise the page

ML approach:
Combine multiple page-level signals to rank pages by refresh priority

Observed model comparison:
Baseline Precision@50: 0.240
Random forest Precision@50: 0.740
The learned model improves the top-50 ranking over the fixed-rule baseline.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.

**Before submission:** run the notebook from top to bottom, confirm there are no errors, save the executed notebook, commit it to `work/notebooks/w02_ml_task_framing.ipynb`, and push the commit.